In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, LeaveOneGroupOut
import homoglyphs as hg
import re
import string
import unicodedata
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [2]:
LING_SPAM_PATH = './datasets/Ling-Spam.csv'
NAZARIO_PATH = './datasets/Nazario.csv'
NIGERIAN_FRAUD_PATH = './datasets/Nigerian_Fraud.csv'
SPAMASSASSIN_PATH = './datasets/SpamAssasin.csv'

REQUIRED_COLUMNS = ['body', 'label']

DATASETS = {
    "Ling-Spam": LING_SPAM_PATH,
    "Nazario": NAZARIO_PATH, # not using Nazario_5
    "Nigerian_Fraud": NIGERIAN_FRAUD_PATH,
    "SpamAssasin": SPAMASSASSIN_PATH
}

# Set random seed for reproducibility
np.random.seed(42)

# Load the dataset individually
#df = pd.read_csv(DATASETS["Ling-Spam"])

#df.head()

In [3]:
# Check for missing values in each dataset and the different datatypes of the columns
for index, name in DATASETS.items():
    print(f"Dataset: {name}")
    df = pd.read_csv(DATASETS[index])
    print(f"{df.isnull().sum()} + \n")
    print(f"{df.info()}" + "\n")

    # Body and label columns are the only common columns across all datasets, so we will only keep those columns for further analysis

Dataset: ./datasets/Ling-Spam.csv
subject    62
message     0
label       0
dtype: int64 + 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   subject  2831 non-null   object
 1   message  2893 non-null   object
 2   label    2893 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 67.9+ KB
None

Dataset: ./datasets/Nazario.csv
sender       0
receiver    96
date         1
subject      4
body         0
urls         0
label        0
dtype: int64 + 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1565 entries, 0 to 1564
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   sender    1565 non-null   object
 1   receiver  1469 non-null   object
 2   date      1564 non-null   object
 3   subject   1561 non-null   object
 4   body      1565 non-null   object
 5   urls      1565 non-null   in

In [4]:
# Check the distribution of labels in each dataset
for index, name in DATASETS.items():
    print(f"Dataset: {name}")
    df = pd.read_csv(DATASETS[index])
    print(f"{df['label'].value_counts(dropna=False)}" + "\n")

Dataset: ./datasets/Ling-Spam.csv
label
0    2412
1     481
Name: count, dtype: int64

Dataset: ./datasets/Nazario.csv
label
1    1565
Name: count, dtype: int64

Dataset: ./datasets/Nigerian_Fraud.csv
label
1    3332
Name: count, dtype: int64

Dataset: ./datasets/SpamAssasin.csv
label
0    4091
1    1718
Name: count, dtype: int64



In [5]:
# Cleaning the SpamAssassin dataset
def clean_spam_assassin(spam_assassin_df):
    if 'body' in spam_assassin_df.columns:
        spam_assassin_df['body'] = spam_assassin_df['body'].fillna('')
    return spam_assassin_df


# Cleaning the Ling-Spam dataset
def clean_ling_spam(ling_spam_df):
    if 'message' in ling_spam_df.columns:
        ling_spam_df = ling_spam_df.rename(columns={'message': 'body'})
    return ling_spam_df



In [6]:
raw_datasets = {}
cleaned_datasets = {}
def preprocess_datasets():

    for name, path in DATASETS.items():
        df = pd.read_csv(path)
        raw_datasets[name] = df

    for name, df in raw_datasets.items():
        df = clean_spam_assassin(df) if name == "SpamAssasin" else df
        df = clean_ling_spam(df) if name == "Ling-Spam" else df

        for column in REQUIRED_COLUMNS:
            if column not in df.columns:
                raise ValueError(f"Column '{column}' is missing in dataset '{name}'")

            cleaned_datasets[name] = df[REQUIRED_COLUMNS].copy()

    for name, df in cleaned_datasets.items():
        print(f"Dataset: {name}")
        print(f"{df.info()}" + "\n")

preprocess_datasets()


Dataset: Ling-Spam
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   body    2893 non-null   object
 1   label   2893 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 45.3+ KB
None

Dataset: Nazario
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1565 entries, 0 to 1564
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   body    1565 non-null   object
 1   label   1565 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 24.6+ KB
None

Dataset: Nigerian_Fraud
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3332 entries, 0 to 3331
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   body    3332 non-null   object
 1   label   3332 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 52.2+ KB
None

Dataset: SpamAssasi

In [7]:
# Training datasets for cross-dataset evaluation design using leave-one-dataset-out evaluation strategy. The training datasets are created by combining three datasets and the testing dataset is the remaining one.

# 1. Train on Nigerian_Fraud + SpamAssasin + Ling-Spam, test on Nazario

# 2. Train on Nigerian_Fraud + SpamAssasin + Nazario, test on Ling-Spam

# 3. Train on Nigerian_Fraud + Ling-Spam + Nazario, test on SpamAssasin

# 4. Train on SpamAssasin + Ling-Spam + Nazario, test on Nigerian_Fraud

# Using a pipeline with TfidfVectorizer and XGBClassifier for the baseline model.
baseline_model = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(3, 5), analyzer='char_wb')),
        ('classifier', XGBClassifier(random_state=42))
    ], tree_method="hist", max_bin=128, learning_rate=0.1)

combined_datasets = pd.concat([df.assign(source=name) for name, df in cleaned_datasets.items()])

logo = LeaveOneGroupOut()
train_df = None
test_df = None
val_df = None
baseline_results = []

for train_idx, test_idx in logo.split(combined_datasets, combined_datasets['label'], groups=combined_datasets['source']):
    training_pool = combined_datasets.iloc[train_idx]
    test_df = combined_datasets.iloc[test_idx]
    held_out_dataset_name = test_df['source'].iloc[0]

    # validation split for training pool
    train_df, val_df = train_test_split(training_pool, test_size=0.2, random_state=42, stratify=training_pool['label'].astype(str) + '_' + training_pool['source'])

    # train the classifier
    #l_rate = [0.01, 0.02, 0.05]
    #max_leaves = [0, 42]

    baseline_model.fit(train_df['body'], train_df['label'])

    pred = baseline_model.predict(test_df['body'])

#    print(f"Training on: {training_pool['source'].unique()} | Testing on: {held_out_dataset_name}")

    print(f"\nTest dataset: {held_out_dataset_name}")
    print(f"Accuracy: {accuracy_score(test_df['label'], pred):.4f}")
    print(classification_report(
        test_df["label"],
        pred,
        zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(test_df["label"], pred))

    baseline_results.append({
        "test_dataset": held_out_dataset_name,
        "accuracy": accuracy_score(test_df["label"], pred)
    })

results_df = pd.DataFrame(baseline_results)




#train_df

#combined_datasets
testing_datasets = cleaned_datasets.copy()

KeyboardInterrupt: 

In [ ]:
label = r"[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?"
segment = r"[A-Za-z0-9]+(?:\s*[._-]\s*[A-Za-z0-9]+)*"

url_pattern = re.compile(
    rf"""
    (?:
        https?\s*:\s*/\s*/
        |
        www\s*\.\s*
    )
    {label}
    (?:\s*\.\s*{label})+
    (?:\s*/\s*{segment})*
    (?:\s*[?&]\s*[A-Za-z0-9._~:/?#\[\]@!$'()*+,;=%-]+)?
    """,
    re.IGNORECASE | re.VERBOSE
    )

def normalize_url(url):
    url = re.sub(r"\s*([:/?.=&])\s*", r"\1", url)
    return url.rstrip(".,;:!?)]}")


# Function to extract URLs in the email body
def extract_url(text):
    normalized_url = [normalize_url(url_match.group(0)) for url_match in url_pattern.finditer(str(text))]
    print(normalized_url)
    return normalized_url

In [ ]:
# Perturbation functions for homoglyph substitution, and encoding obfuscation. These functions will be used to generate adversarial examples for the email text data in the respective test datasets.

# Create a Homoglyphs object with the desired categories
homoglyphs = hg.Homoglyphs(categories=("LATIN", "CYRILLIC"))

latin_letters = string.ascii_letters

# map of similar looking latin letters to cyrillic letters
latin_to_cyrillic = {}

for latin in latin_letters:
    combinations = homoglyphs.get_combinations(latin)

    for c in combinations:
        name = unicodedata.name(c, "")
        if name and "CYRILLIC" in name:
            latin_to_cyrillic[latin] = c
            break

print(latin_to_cyrillic)

def replace_url(m):
    url = m.group(0)
    # Replace each character in the URL with its cyrillic equivalent, if available
    replaced_url = "".join(latin_to_cyrillic.get(char, char) or char for char in url)
    return replaced_url

def homoglyph_substitution(text):
    url = str(text)
    substituted_text = url_pattern.sub(replace_url, url)

    return substituted_text

In [ ]:
#homoglyph_substitution("Hello, check out this link: http : / / www . 123net . co . uk / agents / prebuild . htm and also visit www.testsite.org for more info.")

In [ ]:
# Add perturbation to test datasets
#perturbed_testing_datasets = {}
#for name, df in testing_datasets.items():
#    df['body_homoglyph'] = (df['body'].fillna("").apply(homoglyph_substitution))
#    perturbed_testing_datasets[name] = df
#    print(f"{name}: {df.shape}")


In [ ]:
# first test performance on the original test datasets, then on the perturbed test datasets. The training datasets remain unchanged. this is to have a baseline performance on the original datasets to compare with.
